<a href="https://colab.research.google.com/github/bizz-aa/bizz-simple-page/blob/main/spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
%%writefile test_churn.py
import pytest
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
from pyspark.sql.functions import lit # Added this import
@pytest.fixture(scope="session")
def spark():
  return (SparkSession.builder
          .appName("PysparkTesting")
          .master("local[*]")
          .getOrCreate())
class ChurnFeatureEngineering:
  def __init__(self,target_cities):
    self.target_cities= target_cities
  def transform(self,df):
    if "city" not in df.columns:
      raise ValueError("City coloumn is not found in data")
    from pyspark.sql.functions import col,when
    return df.withColumn(
        "city_grouped",
        when(col("city").isin(self.target_cities), col("city")).otherwise(lit("Other")) # Changed to lit("Other")
        )
def test_processor_city_grouping(spark):
  data=[("C001","Dar"),("C002","unkwown_region")]
  df=spark.createDataFrame(data,["customer_id","city"])
  processor=ChurnFeatureEngineering(target_cities=["Dar","Arusha"])
  result_df=processor.transform(df)
  result_collected=result_df.collect()
  result={r["customer_id"]:r["city_grouped"]for r in result_collected} # Fixed typo: costumer_id to customer_id
  assert result["C001"]=="Dar"
  assert result["C002"]=="Other"

def test_processor_missing_column_exception(spark):
  faulty_data=[("C001",)]

  input_df=spark.createDataFrame(faulty_data,["customer_id"])
  processor=ChurnFeatureEngineering(target_cities=["Dar"])
  with pytest.raises(ValueError,match="City coloumn is not found in data"):
      processor.transform(input_df).collect()

Overwriting test_churn.py


In [32]:
!pytest test_churn.py

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.11.1, typeguard-4.6.0, anyio-4.14.2
collected 2 items                                                              

test_churn.py ..                                                         [100%]

============================== 2 passed in 19.38s ==============================
